# TP2 · 01 — Extração completa: características do TP1 + volumes para as redes (Kaggle, CPU)

Uma única leitura de cada DICOM gera, para **todos os exames do treino oficial (7.279) e do teste
oficial (650)** e **todos os cortes** (sem `slice_stride`):

1. `slices_*.parquet` — os descritores do TP1 por corte, para retreinar o baseline clássico no
   treino completo (exigência da Seção 2 do TP2), e o índice de cada corte (`vol_idx`, `z_rank`,
   `source`, `status`);
2. `volumes_*.npz` — cada exame como volume `uint8` (n_cortes × 256 × 256) na janela CTPA
   (L100/W700), ordenado por z. É a entrada das redes do TP2 (3 canais = cortes adjacentes, 2.5D).

Cortes ilegíveis viram linha com `status = error: ...` e fatia zerada (lista de excluídos).

**Custo estimado:** ≈ 34 h de CPU no total. Use **10 shards** (`SHARD` = 0..9): cada um ≈ 3–4 h e
≈ 6–8 GB de saída (limite de 20 GB por notebook). O Kaggle roda até 5 sessões de CPU ao mesmo
tempo, então são duas levas de 5. Dentro de uma sessão o processo é retomável.

Ao final: **Save Version → Save & Run All**. Cada saída vira um Dataset; os notebooks seguintes
usam as 10 saídas como Input.


In [ ]:
# === Setup (Kaggle) ==========================================================
# Pré-requisitos: aceitar as regras da competição "RSNA STR Pulmonary Embolism Detection"
# e adicioná-la como Input deste notebook (+ Add Input -> Competitions).
import glob, os, sys, subprocess, shutil
REPO_URL = "https://github.com/silvarawenne/ap2-topicos.git"
REPO_BRANCH = "claude/peaceful-curie-mqrljz"   # troque para main após o merge
REPO = "/kaggle/working/ap2-topicos"

if not os.path.exists(REPO):
    # opção sem Internet: o repositório enviado como Dataset (tem configs/config.yaml na raiz)
    hits = glob.glob("/kaggle/input/**/configs/config.yaml", recursive=True)
    if hits:
        shutil.copytree(os.path.dirname(os.path.dirname(hits[0])), REPO)
    else:  # exige "Internet on" nas configurações do notebook
        subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, REPO], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO}/requirements-kaggle.txt"], check=True)
os.chdir(REPO)
sys.path.insert(0, f"{REPO}/src")
# os dados da competição são achados sozinhos (ou defina PE_DATA_ROOT)
hits = glob.glob("/kaggle/input/**/train.csv", recursive=True)
if hits:
    os.environ.setdefault("PE_DATA_ROOT", os.path.dirname(hits[0]))
print("repo:", REPO, "| dados:", os.environ.get("PE_DATA_ROOT"))


In [ ]:
# Escolha o shard desta cópia: 0..9 (LIMIT = 3 faz um teste rápido com 3 exames)
SHARD, N_SHARDS, LIMIT = 0, 10, None
OUT = "/kaggle/working/features_full"
os.environ["PE_FEATURES_DIR"] = OUT
extra = f"--limit {LIMIT} --chunk-size 3" if LIMIT else ""
!python scripts/01_make_split_and_sample.py
!python -W ignore scripts/02_extract_features.py --full --shard {SHARD} --n-shards {N_SHARDS} --out-dir {OUT}/slices {extra}

In [ ]:
import pandas as pd
files = sorted(glob.glob(f"{OUT}/slices/*.parquet"))
df = pd.concat([pd.read_parquet(f, columns=["StudyInstanceUID", "source", "status", "vol_file"]) for f in files])
print(len(files), "blocos |", len(df), "cortes |", df.StudyInstanceUID.nunique(), "exames")
print(df.groupby("source").StudyInstanceUID.nunique())
print(df.status.str.slice(0, 60).value_counts().head())
print("exames sem volume:", df.loc[df.vol_file == "", "StudyInstanceUID"].nunique())

In [ ]:
# A partição acompanha a saída; tamanho total (deve ficar < 20 GB)
!cp -r splits {OUT}/ && du -sh {OUT} && ls {OUT}/slices | wc -l